# P3 — Task 3 — Engagement + soát số toàn deck

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanh-tai-435/BI10GroupYAPPERS/blob/main/notebooks/task3_engagement.ipynb)

**Slide sở hữu:** 11, 12, 13 · **Slide mới:** 0 (làm chart tốt hơn) + soát số  
**Đã có:** task3.py, task3_findings.md, chart t3_engagement_dist / t3_channel_mix / t3_diversity_vs_engagement / t3_health_vs_engagement_quadrant

**3 luật:** (1) Wellbeing, KHÔNG credit scoring — FHS không bao giờ dùng để từ chối/giảm hạn mức/khoá. (2) So bằng tỷ lệ, không so VND tuyệt đối. (3) Mỗi nhận định có số hoặc chart.

Code đầy đủ: `draft/task3_deep.py` (chạy độc lập được). Hướng dẫn chi tiết: `draft/YAPPERS_huong_dan_Colab.xlsx`, sheet của bạn.

In [ ]:
#@title ⚙️ SETUP — chạy 1 lần mỗi phiên Colab (không cần pip install gì cả)
# Colab có sẵn pandas / numpy / scikit-learn / matplotlib / openpyxl -> đủ cho mọi script trong draft/.
import os, re, sys, gzip, shutil, urllib.request
REPO = "/content/BI10GroupYAPPERS"
# Data tải thẳng từ thư mục Drive public (anyone with link) — không cần mount Drive / Add shortcut
# Thư mục: https://drive.google.com/drive/folders/1gOqCyY9IfFUJtbr2r3vR_Z3t61O71CrC
URL = {
    "consumer_financial_health_engagement_2025.csv": "https://drive.google.com/file/d/1YyueAH9Kasid6V_obml9pCMwh2dB0ryn/view",
    "consumer_transactions_2025.parquet.gz":         "https://drive.google.com/file/d/1O33KpFbUx6_BRGwFTb80xVSB3tzXqt5H/view",
    "data_dictionary.xlsx":                          "https://drive.google.com/file/d/1nFlZHmuaoK8VQ5xqRN2f4ZLqoR3gk2j1/view",
}

from google.colab import userdata

if os.path.exists(REPO):
    os.system(f"git -C {REPO} pull -q")
else:
    tok = userdata.get("GH_TOKEN")   # 🔑 Secrets bên trái: tên GH_TOKEN, bật "Notebook access"
    os.system(f"git clone -q https://{tok}@github.com/thanh-tai-435/BI10GroupYAPPERS.git {REPO}")
assert os.path.exists(REPO), "Clone thất bại: kiểm tra GH_TOKEN + đã được mời vào repo chưa"
os.chdir(REPO)

# Dựng lại đúng đường dẫn mà các script draft/*.py đang dùng
D = "BI10_ROUND01_DATASET"
TX = f"{D}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
os.makedirs(os.path.dirname(TX), exist_ok=True)
for f, u in URL.items():
    if not os.path.exists(f"{D}/{f}"):
        fid = re.search(r"/d/([\w-]+)", u).group(1)   # tải file public bằng stdlib, không cần gdown
        urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", f"{D}/{f}")
if not os.path.exists(TX):   # .parquet.gz thực chất là CSV giao dịch nén gzip (134MB -> 597MB, ~30s)
    with gzip.open(f"{D}/consumer_transactions_2025.parquet.gz") as src, open(TX, "wb") as dst:
        shutil.copyfileobj(src, dst)

# Script gốc gọi sys.stdout.reconfigure(); stdout của Colab không có hàm này -> cho thành no-op
if not hasattr(sys.stdout, "reconfigure"):
    type(sys.stdout).reconfigure = lambda self, **k: None
os.makedirs("outputs/figures", exist_ok=True)
if not os.path.exists("draft/task4_features.csv"):   # nhãn segment (git-ignore) — task3/task5/charts cần
    os.system("python draft/task4.py > /dev/null")
print("✅ Sẵn sàng — cwd:", os.getcwd())


## Bước 0 — Chạy baseline
Nạp toàn bộ biến của script gốc (`mon`, `cons`, `txn`…) vào notebook. So kết quả với `draft/task3_output.txt`.

In [ ]:
%run -i draft/task3.py

In [ ]:
# Style chung cho chart (giống draft/make_charts.py)
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 250, "display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 11, "axes.titlesize": 15, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
C = dict(primary="#2E5EAA", accent="#E4572E", good="#3C9D6E", warn="#E0A32E", muted="#9AA5B1", ink="#22303F")
os.makedirs("outputs/figures", exist_ok=True)
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"outputs/figures/{name}", dpi=150, bbox_inches="tight"); plt.show(); print("saved", name)
SEG_EN = {"Financially Healthy & Highly Engaged": "Healthy & Engaged", "Financially Stretched but Highly Engaged": "Stretched & Engaged",
          "High-Activity Digital Power Users": "Digital Power Users", "Low Engagement & Emerging Digital": "Emerging Digital"}
ENG_EN = {"Tương tác thấp": "Low", "Tương tác trung bình": "Medium", "Tương tác cao": "High", "Tương tác rất cao": "Very high"}


## Việc 1 — Deliverable 1 — % khách từng engagement segment  `[BẮT BUỘC · hạn 27/09 12:00]`
Không chỉ '99% high/very high': ghi % từng nhóm lên histogram (nhãn trên cột).

**Giao:** Chart thay thế slide 11

In [ ]:
h("DEEP D1 - Share of consumers in EACH engagement segment (consumer = modal segment)")
share = modal.map(ENG_EN).value_counts().reindex(["Low", "Medium", "High", "Very high"])
print(pd.DataFrame({"consumers": share, "pct": (100 * share / N).round(1)}).to_string())
fig, ax = plt.subplots(figsize=(9, 5))
b = ax.bar(share.index, share.values, color=[C["accent"], C["warn"], C["primary"], C["good"]])
for r, v in zip(b, share.values):
    ax.annotate(f"{v} ({100*v/N:.1f}%)", (r.get_x() + r.get_width()/2, v), xytext=(0, 3), textcoords="offset points", ha="center")
ax.set_ylabel("Consumers (modal monthly segment)"); ax.set_xlabel("Engagement segment")
ax.set_title("91% of consumers sit in High / Very-high engagement")
save(fig, "t3_segment_share.png")


## Việc 2 — Deliverable 2 — average online spend share + ai dùng QR/Mobile  `[Cao · hạn 27/09 12:00]`
Nêu rõ TB online_spend_ratio cấp khách trên slide 11. Thêm kênh theo nhóm tuổi hoặc theo segment của P4 (nhãn trong draft/task4_features.csv).

**Giao:** Bullet slide 11 + gửi P5 (Reminders/Nudges)

In [ ]:
h("DEEP D2 - Average online spend share + who uses QR / Mobile (by Task 4 segment)")
print(f"Average online spend share (consumer level): {100*con.online_spend_ratio.mean():.1f}% "
      f"(month level {100*mon.online_spend_ratio.mean():.1f}%)")
seg = pd.read_csv("draft/task4_features.csv", usecols=["consumer_id", "segment"])
tx = pd.read_csv(TXN, usecols=["consumer_id", "transaction_channel", "spend_amount_vnd"]).merge(seg, on="consumer_id")
chs = pd.crosstab(tx.segment.map(SEG_EN), tx.transaction_channel, normalize="index") * 100
chv = pd.crosstab(tx.segment.map(SEG_EN), tx.transaction_channel, values=tx.spend_amount_vnd, aggfunc="sum", normalize="index") * 100
print("Channel share of TRANSACTIONS by segment (%):\n", chs.round(1).to_string())
print("\nChannel share of SPEND by segment (%):\n", chv.round(1).to_string())


## Việc 3 — Deliverable 3 — histogram diversity + kiểm r = +0.94  `[Trung bình · hạn 27/09 18:00]`
Histogram category diversity (2–14) cạnh scatter. Tính lại r chỉ trên khách diversity ≥ 8 để xem có bị đuôi nhỏ kéo không.

**Giao:** Chart + bullet slide 12

In [ ]:
h("DEEP D3 - Category diversity distribution + is r=+0.94 driven by the small tail?")
cdv = con.category_diversity
print("Consumer-level diversity bins:\n", pd.cut(cdv, [0, 4, 6, 8, 10, 12, 13, 14.01]).value_counts().sort_index().to_string())
core = con[con.category_diversity >= 8]
print(f"r(all {len(con)}) = {con.category_diversity.corr(con.engagement_score):+.3f} | "
      f"r(diversity>=8, n={len(core)}) = {core.category_diversity.corr(core.engagement_score):+.3f} | "
      f"Spearman(all) = {con.category_diversity.corr(con.engagement_score, method='spearman'):+.3f}")
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(cdv, bins=np.arange(1.5, 14.6, 0.5), color=C["primary"], edgecolor="white")
ax.set_yscale("log"); ax.set_xlabel("Category diversity (consumer mean of monthly count, max 14)")
ax.set_ylabel("Consumers (log scale)"); ax.set_title("Diversity: a saturated core near 14 and a thin dormant tail")
save(fig, "t3_diversity_hist.png")


## Việc 4 — Deliverable 4 — heatmap Recency × Frequency  `[Cao · hạn 27/09 18:00]`
Bin recency × bin active_transaction_days, màu = engagement TB. 'Phù hợp dữ liệu' hơn bảng số (Objective 6).

**Giao:** Chart slide 12

In [ ]:
h("DEEP D4 - Recency x Frequency heatmap (consumer-month, colour = mean engagement)")
rb = pd.cut(mon.transaction_recency_days, [-0.1, 0, 3, 10, 31], labels=["0 d", "1-3 d", "4-10 d", "11-30 d"])
fb = pd.cut(mon.active_transaction_days, [0, 5, 15, 25, 29, 31], labels=["1-5", "6-15", "16-25", "26-29", "30-31"])
hm = mon.pivot_table(index=rb, columns=fb, values="engagement_score", aggfunc="mean", observed=False)
hn = mon.pivot_table(index=rb, columns=fb, values="engagement_score", aggfunc="size", observed=False)
print("Mean engagement:\n", hm.round(1).to_string()); print("\nConsumer-months per cell:\n", hn.fillna(0).astype(int).to_string())
fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.imshow(hm.values, cmap="YlGnBu", vmin=20, vmax=85, aspect="auto")
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        n = hn.iloc[i, j]
        if n > 0 and not np.isnan(hm.iloc[i, j]):
            ax.text(j, i, f"{hm.iloc[i, j]:.0f}\nn={int(n):,}", ha="center", va="center", fontsize=9,
                    color="white" if hm.iloc[i, j] > 60 else C["ink"])
ax.set_xticks(range(hm.shape[1]), hm.columns); ax.set_yticks(range(hm.shape[0]), hm.index)
ax.set_xlabel("Active transaction days in month (frequency)"); ax.set_ylabel("Days since last transaction (recency)")
ax.grid(False); fig.colorbar(im, ax=ax, label="Mean engagement score")
ax.set_title("Engagement is driven by consistent activity, not recency alone")
save(fig, "t3_recency_frequency.png")


## Việc 5 — Deliverable 5 — bảng độ nhạy cutoff Healthy-but-Disengaged  `[BẮT BUỘC · hạn 27/09 18:00]`
Số khách khi FHS ≥ 65/70/75 × engagement < 60/65/70/75. Đề hỏi 'vì sao chọn cutoff này thay vì chặt/lỏng hơn' → bảng này là câu trả lời.

**Giao:** Bảng slide 13

In [ ]:
h("DEEP D5 - Healthy-but-Disengaged cutoff sensitivity (consumer level)")
tab = pd.DataFrame({f"eng<{e}": [int(((con.financial_health_score >= f) & (con.engagement_score < e)).sum()) for f in (65, 70, 75)]
                    for e in (60, 65, 70, 75)}, index=[f"FHS>={f}" for f in (65, 70, 75)])
print(tab.to_string())
gap = con.engagement_score.sort_values()
print("Engagement values around the cutoff (consumer level): largest gap between consecutive consumers in 50-75:",
      gap[(gap > 50) & (gap < 75)].diff().max().round(2))
print("Why FHS>=70 & eng<70: the count is flat (25) from eng<60 to eng<70 -> a natural gap; eng<75 jumps as it hits the main body.")


## Việc 6 — 1 dòng 'vì sao chọn loại chart này' cho mỗi chart  `[Trung bình · hạn 27/09 18:00]`
Ghi trong phần Findings của notebook → P5 đưa vào speaker note.

**Giao:** Speaker note

In [ ]:
h("Chart-choice rationale (for speaker notes)")
for k, v in {
    "t3_segment_share.png": "Bar of 4 ordered categories with counts + % -> reads shares exactly (pie hides small tails).",
    "t3_diversity_hist.png": "Histogram, log y: continuous variable with a huge spike at 14 and a thin tail -> log keeps the tail visible.",
    "t3_recency_frequency.png": "Heatmap: two binned drivers x one outcome -> shows the interaction a pair of scatters would hide.",
    "t3_diversity_vs_engagement.png": "Scatter: two continuous consumer-level variables, correlation is the claim.",
    "t3_channel_mix.png": "Bar by channel: 5 nominal categories, count vs value compared side by side.",
}.items():
    print(f"  {k:<34} {v}")


## Việc 7 — SOÁT SỐ TOÀN DECK (việc thêm để cân tải)  `[Cao · hạn 28/09 12:00]`
Khi P5 gửi deck v2 (27/09 23:00): so MỌI con số trên slide 2, 16, 17, 18 với output các task (chạy lại %run từng script nếu cần). Lập danh sách lệch → gửi P5 trước 28/09 12:00.

**Giao:** Danh sách lệch số → P5

In [ ]:
# Việc không cần code phân tích — gợi ý:
# —


## Findings — kết quả đã chạy 27/09 (chi tiết: `draft/task3_deep_output.txt`)
- D1: khách theo segment: Low **1.0%** · Medium **8.0%** · High **74.1%** · Very high **16.9%**.
- D2: online spend share TB **24.7%** (cấp khách). Emerging Digital là nhóm dùng Mobile/E-com: **60.5% giá trị** qua E-com+Mobile (3 segment còn lại ~17%).
- D3: diversity dồn ở 13–14 (908/999), đuôi ≤8 chỉ 91 khách; r bỏ đuôi (diversity ≥8) vẫn **+0.81** (Spearman toàn bộ +0.85) → quan hệ không chỉ do đuôi kéo.
- D4: heatmap: engagement cao nhất ở ô hoạt động 30–31 ngày & recency 0 (79.8, n=6,614); 1–5 ngày hoạt động luôn ≤55 bất kể recency → tần suất đều đặn quan trọng hơn recency.
- D5: FHS≥70 & eng<70 = **25** khách, không đổi khi eng<60/65/70 (khoảng trống tự nhiên), nhảy lên 63 ở eng<75 → cutoff 70 đúng chỗ.

---
**Lưu bài:** `File → Save a copy in GitHub` → repo `thanh-tai-435/BI10GroupYAPPERS`, branch `main`, path `notebooks/task3_engagement.ipynb` (mỗi người chỉ lưu notebook của mình → không conflict). Chart: `savefig('outputs/figures/tN_ten.png')` rồi tải về gửi P5.